# LLM-Augmented Stock Return Prediction — Analysis

**Course:** AI & Machine Learning, Copenhagen Business School  
**Research question:** Does providing an LLM with financial news improve next-day stock return direction prediction compared to price data alone — and how do LLM-based approaches compare to a traditional LSTM baseline?

| Condition | Description | Method |
|-----------|-------------|--------|
| **B**  | LSTM baseline | LSTM on OHLCV + technical indicators → log return |
| **L1** | LLM — price only | GPT-5-nano on last 10 days of price data → up/down |
| **L2** | LLM — news only | GPT-5-nano on top-5 article summaries → up/down |
| **L3** | LLM — price + news | GPT-5-nano on price + news → up/down |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ROOT = os.path.join(os.getcwd(), '..')

prices   = pd.read_csv(f'{ROOT}/data/processed/prices.csv', parse_dates=['date'], index_col='date')
news     = pd.read_csv(f'{ROOT}/data/processed/news_cache.csv', parse_dates=['date'])
pred_B   = pd.read_csv(f'{ROOT}/results/predictions_B.csv',  parse_dates=['date'])
pred_L1  = pd.read_csv(f'{ROOT}/results/predictions_L1.csv', parse_dates=['date'])
pred_L2  = pd.read_csv(f'{ROOT}/results/predictions_L2.csv', parse_dates=['date'])
pred_L3  = pd.read_csv(f'{ROOT}/results/predictions_L3.csv', parse_dates=['date'])

print('prices  :', prices.shape,  '|', prices.index[0].date(), '–', prices.index[-1].date())
print('news    :', news.shape)
print('pred B  :', pred_B.shape)
print('pred L1 :', pred_L1.shape)
print('pred L2 :', pred_L2.shape)
print('pred L3 :', pred_L3.shape)

---
## Section 1 — Data Overview

In [ ]:
# ── AAPL price history: close + MA5 + MA20 ───────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(prices.index, prices['close'], color='steelblue', lw=1.2, label='Close')
ax.plot(prices.index, prices['ma5'],  color='orange',    lw=1.0, ls='--', label='MA5')
ax.plot(prices.index, prices['ma20'], color='tomato',    lw=1.0, ls='--', label='MA20')

ax.set_title('AAPL Daily Close Price with Moving Averages (Jul 2024 – Apr 2026)')
ax.set_ylabel('Price (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.legend()
fig.tight_layout()
plt.savefig(f'{ROOT}/results/figures/price_history.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Daily log returns bar chart ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3))

colors = ['tomato' if r < 0 else 'mediumseagreen' for r in prices['daily_log_return']]
ax.bar(prices.index, prices['daily_log_return'], color=colors, width=1.2, alpha=0.8)
ax.axhline(0, color='black', lw=0.5)

ax.set_title('AAPL Daily Log Returns')
ax.set_ylabel('Log Return')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.tight_layout()
plt.savefig(f'{ROOT}/results/figures/log_returns.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── News coverage: articles per trading day ───────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3))

news_plot = news.copy()
news_plot['date'] = pd.to_datetime(news_plot['date'])
news_plot = news_plot.set_index('date').sort_index()

ax.bar(news_plot.index, news_plot['article_count'], color='steelblue', width=1.2, alpha=0.8)
ax.set_title('AAPL News Articles per Trading Day (Massive / Polygon)')
ax.set_ylabel('Article Count')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

pct_covered = 100 * (news_plot['article_count'] > 0).mean()
ax.set_xlabel(f'{pct_covered:.1f}% of trading days have ≥1 article')
fig.tight_layout()
plt.savefig(f'{ROOT}/results/figures/news_coverage.png', bbox_inches='tight')
plt.show()

---
## Section 2 — Walk-Forward Prediction Charts

In [ ]:
# ── Condition B: actual vs predicted log return ───────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(pred_B['date'], pred_B['actual_return'],    color='steelblue', lw=1.2, label='Actual return')
ax.plot(pred_B['date'], pred_B['predicted_return'], color='orange',    lw=1.0, ls='--', alpha=0.8, label='Predicted return')

correct = pred_B['actual_direction'] == pred_B['predicted_direction']
ax.scatter(pred_B.loc[correct, 'date'],  pred_B.loc[correct, 'actual_return'],
           color='mediumseagreen', s=12, zorder=3, label='Correct direction')
ax.scatter(pred_B.loc[~correct, 'date'], pred_B.loc[~correct, 'actual_return'],
           color='tomato',         s=12, zorder=3, label='Wrong direction')

acc = correct.mean()
ax.set_title(f'Condition B (LSTM Baseline) — Directional Accuracy: {acc:.1%}')
ax.set_ylabel('Log Return')
ax.axhline(0, color='black', lw=0.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.legend(fontsize=8)
fig.tight_layout()
plt.savefig(f'{ROOT}/results/figures/pred_B.png', bbox_inches='tight')
plt.show()

In [ ]:
def plot_llm_condition(pred_df, label, ax):
    """Plot LLM up/down predictions with correct (green) / wrong (red) markers."""
    pred_df = pred_df.dropna(subset=['actual_direction', 'predicted_direction'])
    correct  = pred_df['actual_direction'] == pred_df['predicted_direction']
    acc      = correct.mean()

    # Actual direction as +1 / -1 step line
    actual_dir = pred_df['actual_direction'].map({1: 1, 0: -1})
    ax.step(pred_df['date'], actual_dir, where='mid', color='steelblue', lw=0.8, alpha=0.5, label='Actual')

    ax.scatter(pred_df.loc[correct, 'date'],  [1.15] * correct.sum(),
               color='mediumseagreen', s=14, marker='|', label='Correct')
    ax.scatter(pred_df.loc[~correct, 'date'], [-1.15] * (~correct).sum(),
               color='tomato',         s=14, marker='|', label='Wrong')

    ax.set_title(f'Condition {label} — Directional Accuracy: {acc:.1%}')
    ax.set_yticks([-1, 1]); ax.set_yticklabels(['Down', 'Up'])
    ax.axhline(0, color='black', lw=0.4)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    return acc

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, (df, label) in zip(axes, [(pred_L1, 'L1'), (pred_L2, 'L2'), (pred_L3, 'L3')]):
    plot_llm_condition(df, label, ax)

axes[0].legend(fontsize=8, loc='upper right')
fig.tight_layout()
plt.savefig(f'{ROOT}/results/figures/pred_LLM.png', bbox_inches='tight')
plt.show()

---
## Section 3 — Metrics Summary

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
def dir_accuracy(pred_df, act_col='actual_direction', pred_col='predicted_direction'):
    df = pred_df.dropna(subset=[act_col, pred_col])
    return (df[act_col] == df[pred_col]).mean()

def conf_weighted_accuracy(pred_df, threshold=0.7):
    df = pred_df.dropna(subset=['actual_direction', 'predicted_direction', 'confidence'])
    high = df[df['confidence'] >= threshold]
    if len(high) == 0:
        return float('nan'), 0
    return (high['actual_direction'] == high['predicted_direction']).mean(), len(high)

def regression_metrics(pred_df):
    actual = pred_df['actual_return'].values
    pred   = pred_df['predicted_return'].values
    mae    = np.mean(np.abs(actual - pred))
    rmse   = np.sqrt(np.mean((actual - pred) ** 2))
    return mae, rmse

# ── Unified comparison table ──────────────────────────────────────────────────
rows = []

mae_B, rmse_B = regression_metrics(pred_B)
rows.append({
    'Condition': 'B (LSTM baseline)',
    'N days':    len(pred_B),
    'Dir. Acc':  f"{dir_accuracy(pred_B):.1%}",
    'CW Acc (≥0.7)': '—',
    'MAE':       f"{mae_B:.5f}",
    'RMSE':      f"{rmse_B:.5f}",
})

for df, label in [(pred_L1, 'L1 (price only)'), (pred_L2, 'L2 (news only)'), (pred_L3, 'L3 (price+news)')]:
    cw_acc, n_high = conf_weighted_accuracy(df)
    rows.append({
        'Condition': label,
        'N days':    len(df.dropna(subset=['actual_direction'])),
        'Dir. Acc':  f"{dir_accuracy(df):.1%}",
        'CW Acc (≥0.7)': f"{cw_acc:.1%} (n={n_high})",
        'MAE':       '—',
        'RMSE':      '—',
    })

summary = pd.DataFrame(rows).set_index('Condition')
print('Random baseline: 50.0%')
summary

In [ ]:
# ── Confusion matrices for all four conditions ────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

conditions = [
    (pred_B,  'B (LSTM)',        'actual_direction', 'predicted_direction'),
    (pred_L1, 'L1 (price only)', 'actual_direction', 'predicted_direction'),
    (pred_L2, 'L2 (news only)',  'actual_direction', 'predicted_direction'),
    (pred_L3, 'L3 (price+news)', 'actual_direction', 'predicted_direction'),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for ax, (df, label, act_col, pred_col) in zip(axes, conditions):
    df_clean = df.dropna(subset=[act_col, pred_col])
    cm = confusion_matrix(df_clean[act_col].astype(int), df_clean[pred_col].astype(int))
    disp = ConfusionMatrixDisplay(cm, display_labels=['Down', 'Up'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(label, fontsize=10)

fig.suptitle('Confusion Matrices — All Conditions', y=1.02)
fig.tight_layout()
plt.savefig(f'{ROOT}/results/figures/confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Confidence distribution for LLM conditions ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)

for ax, (df, label) in zip(axes, [(pred_L1, 'L1'), (pred_L2, 'L2'), (pred_L3, 'L3')]):
    df_c = df.dropna(subset=['confidence', 'actual_direction', 'predicted_direction'])
    correct = df_c['actual_direction'] == df_c['predicted_direction']
    ax.hist(df_c.loc[correct,  'confidence'], bins=10, alpha=0.7, color='mediumseagreen', label='Correct')
    ax.hist(df_c.loc[~correct, 'confidence'], bins=10, alpha=0.7, color='tomato',         label='Wrong')
    ax.axvline(0.7, color='black', ls='--', lw=1, label='Threshold (0.7)')
    ax.set_title(f'Condition {label}')
    ax.set_xlabel('Confidence')

axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)
fig.suptitle('Confidence Distribution by Prediction Outcome')
fig.tight_layout()
plt.savefig(f'{ROOT}/results/figures/confidence_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── MAE and RMSE for condition B ──────────────────────────────────────────────
print('Condition B — Regression metrics')
print(f'  MAE : {mae_B:.6f}')
print(f'  RMSE: {rmse_B:.6f}')
print(f'  (Naive baseline RMSE — std of actual returns: {pred_B["actual_return"].std():.6f})')

---
## Section 4 — Discussion

### 4.1 Summary of results

*Fill in after all runs complete.*

### 4.2 Does news help the LLM?

Comparing **L1** (price only) with **L2** (news only) and **L3** (price + news) isolates the marginal contribution of financial news. If L2 or L3 outperforms L1, this suggests that same-day news summaries carry predictive signal beyond what is encoded in historical price data.

### 4.3 LLM vs LSTM baseline

All four conditions are evaluated on the same 188 trading days, so directional accuracy is directly comparable. The 50% random baseline serves as the floor; results only slightly above it should be interpreted cautiously given the short evaluation window.

### 4.4 Limitations

- **Sample size:** 188 evaluation days is a short window. A 5 percentage-point difference in directional accuracy (e.g. 55% vs 50%) corresponds to roughly 9 additional correct calls — well within random variation.
- **Non-stationarity:** The evaluation period (Jul 2025 – Apr 2026) spans varying market regimes. Performance may differ materially across sub-periods.
- **News quality:** The Massive API descriptions are short (1–3 sentences). Richer article bodies might provide stronger signal for conditions L2 and L3.
- **LLM temperature:** `gpt-5-nano` does not support `temperature=0`, so LLM predictions carry sampling variance. Running multiple passes per day would reduce this but increases API cost.
- **LSTM size:** The 250-day rolling window leaves limited data for a 2-layer LSTM with seq_len=20. A simpler model or longer history might improve condition B.